# Head-Level Analysis

Attention heads are the fundamental computational unit of transformers.
Understanding what each head does is central to mechanistic interpretability.

This notebook covers the `irtk.head_analysis` module:
- Finding previous-token heads and induction heads
- Scoring heads across multiple importance metrics
- Classifying heads by behavior type
- Clustering heads by behavioral similarity

## Why This Matters

Attention heads are the primary information-routing mechanism in transformers. Classifying heads by function (previous-token, induction, copying, inhibition) is the first step toward understanding what algorithm the model implements for any given task.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk import head_analysis

# Load GPT-2 Small
model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"Model: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads per layer")
print(f"Total heads: {model.cfg.n_layers * model.cfg.n_heads}")

## 1. Finding Previous-Token Heads

A **previous-token head** attends primarily to position i-1 when querying from position i.
These heads shift information one token backward and are a key component of induction circuits.

In [ ]:
tokens = model.to_tokens("The quick brown fox jumped over the lazy dog and then ran away quickly")
_, cache = model.run_with_cache(tokens)

prev_heads = head_analysis.find_previous_token_heads(cache, model, threshold=0.3)
print(f"Found {len(prev_heads)} previous-token heads (threshold=0.3):\n")
for layer, head, score in prev_heads[:10]:
    print(f"  L{layer}H{head}: score = {score:.3f}")

In [ ]:
# Visualize the attention pattern of the top previous-token head
if prev_heads:
    top_l, top_h, top_score = prev_heads[0]
    pattern = np.array(cache[("pattern", top_l)][top_h])
    
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(pattern[:20, :20], cmap="Blues", vmin=0, vmax=1)
    ax.set_title(f"L{top_l}H{top_h} Attention (score={top_score:.3f})")
    ax.set_xlabel("Key position")
    ax.set_ylabel("Query position")
    plt.colorbar(im, ax=ax)
    
    # Highlight the sub-diagonal
    for i in range(1, min(20, pattern.shape[0])):
        ax.plot(i-1, i, 'rx', markersize=8)
    
    plt.tight_layout()
    plt.show()

## 2. Finding Induction Heads

An **induction head** implements the pattern: when it sees `[A][B]...[A]`, it predicts `[B]`.
It attends to the token that followed the previous occurrence of the current token.

Detection uses a repeated random sequence: the model sees `[random tokens][same random tokens]`,
and we check if heads in the second half attend to the right position in the first half.

In [ ]:
induction_heads = head_analysis.find_induction_heads(model, seq_len=30, threshold=0.2)
print(f"Found {len(induction_heads)} induction heads (threshold=0.2):\n")
for layer, head, score in induction_heads[:10]:
    print(f"  L{layer}H{head}: score = {score:.3f}")

In [ ]:
# Induction heads are typically in layer 1+ (they need previous-token heads in layer 0)
if induction_heads:
    ind_layers = [l for l, h, s in induction_heads]
    print(f"Induction head layers: {sorted(set(ind_layers))}")
    print(f"Most induction heads in layer(s): ", end="")
    from collections import Counter
    layer_counts = Counter(ind_layers)
    print(layer_counts.most_common(3))

## 3. Head Importance Scores

`head_importance_scores` computes multiple metrics for every head at once:
- **Entropy**: How broadly the head attends (high = broad, low = focused)
- **Max attention**: Average of the strongest attention weight per query
- **Previous-token score**: How much it attends to position i-1
- **Diagonal score**: How much it attends to position i (self-attention)
- **Direct logit attribution**: (optional) How much the head contributes to a specific token's logit

In [ ]:
scores = head_analysis.head_importance_scores(model, tokens)
print("Metrics computed:")
for key, val in scores.items():
    print(f"  {key}: shape {val.shape}, range [{val.min():.3f}, {val.max():.3f}]")

In [ ]:
# Visualize all metrics as heatmaps
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
metrics = ["entropy", "max_attn", "prev_token", "diag_score"]
cmaps = ["YlOrRd", "Blues", "Greens", "Purples"]

for ax, metric, cmap in zip(axes.flat, metrics, cmaps):
    im = ax.imshow(scores[metric], aspect="auto", cmap=cmap)
    ax.set_title(metric.replace("_", " ").title())
    ax.set_xlabel("Head")
    ax.set_ylabel("Layer")
    plt.colorbar(im, ax=ax)

plt.suptitle("Head Importance Scores", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# With direct logit attribution for a specific token
target_token = tokenizer.encode(" dog")[0]
scores_dla = head_analysis.head_importance_scores(model, tokens, target_token=target_token)
print(f"Direct logit attribution for ' dog' (token {target_token}):")
print(f"  Shape: {scores_dla['direct_logit'].shape}")

# Show top heads for this token
dla = scores_dla["direct_logit"]
flat_idx = np.argsort(dla.flatten())[::-1]
print("  Top 5 heads:")
for idx in flat_idx[:5]:
    l, h = divmod(idx, model.cfg.n_heads)
    print(f"    L{l}H{h}: {dla[l, h]:.4f}")

## 4. Classifying Heads

`classify_heads` assigns each head to one of five categories:
- **previous_token**: Attends to i-1
- **self_attention**: Attends to i (diagonal)
- **bos_attention**: Attends to position 0 (BOS/first token)
- **broad**: High entropy (attends broadly across all positions)
- **other**: Doesn't fit the above

In [ ]:
categories = head_analysis.classify_heads(model, tokens)

print("Head Classification:")
for cat, heads in categories.items():
    print(f"  {cat}: {len(heads)} heads")
    if heads:
        print(f"    Examples: {heads[:5]}")

In [ ]:
# Visualize classification as a grid
color_map = {
    "previous_token": 0,
    "self_attention": 1,
    "bos_attention": 2,
    "broad": 3,
    "other": 4,
}
label_names = list(color_map.keys())

grid = np.full((model.cfg.n_layers, model.cfg.n_heads), -1)
for cat, heads in categories.items():
    for l, h in heads:
        grid[l, h] = color_map[cat]

fig, ax = plt.subplots(figsize=(10, 8))
cmap = plt.cm.get_cmap("Set2", 5)
im = ax.imshow(grid, cmap=cmap, vmin=-0.5, vmax=4.5, aspect="auto")
ax.set_xlabel("Head")
ax.set_ylabel("Layer")
ax.set_title("Head Classification")

cbar = plt.colorbar(im, ax=ax, ticks=range(5))
cbar.ax.set_yticklabels(label_names)

plt.tight_layout()
plt.show()

## 5. Head Clustering

`head_clustering` groups heads by behavioral similarity using k-means on a feature vector
of (entropy, max_attn, prev_token, diag_score). This discovers natural groupings that
may not correspond to the rule-based classification above.

In [ ]:
cluster_result = head_analysis.head_clustering(model, tokens, n_clusters=5)
print(f"Feature names: {cluster_result['feature_names']}")
print(f"Features shape: {cluster_result['features'].shape}")
print(f"Labels shape: {cluster_result['labels'].shape}")
print(f"\nCluster distribution:")
for k in range(5):
    count = np.sum(cluster_result["labels"] == k)
    print(f"  Cluster {k}: {count} heads")

In [ ]:
# Visualize clusters as a grid
labels = cluster_result["labels"].reshape(model.cfg.n_layers, model.cfg.n_heads)

fig, ax = plt.subplots(figsize=(10, 8))
cmap = plt.cm.get_cmap("tab10", 5)
im = ax.imshow(labels, cmap=cmap, vmin=-0.5, vmax=4.5, aspect="auto")
ax.set_xlabel("Head")
ax.set_ylabel("Layer")
ax.set_title("Head Clusters (k-means, k=5)")
plt.colorbar(im, ax=ax, label="Cluster")

plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot of features colored by cluster
features = cluster_result["features"]
flat_labels = cluster_result["labels"]
names = cluster_result["feature_names"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

pairs = [(0, 1), (0, 2), (2, 3)]  # Feature pairs to plot
for ax, (i, j) in zip(axes, pairs):
    for k in range(5):
        mask = flat_labels == k
        ax.scatter(features[mask, i], features[mask, j], s=20, alpha=0.6, label=f"Cluster {k}")
    ax.set_xlabel(names[i])
    ax.set_ylabel(names[j])
    ax.legend(fontsize=7)

plt.suptitle("Head Features by Cluster")
plt.tight_layout()
plt.show()

## Summary

| Function | Purpose |
|----------|--------|
| `find_previous_token_heads()` | Find heads that attend to position i-1 |
| `find_induction_heads()` | Find heads that implement the [A][B]...[A] -> [B] pattern |
| `head_importance_scores()` | Compute entropy, max_attn, prev_token, diag_score, direct_logit per head |
| `classify_heads()` | Classify heads as previous_token / self_attention / bos_attention / broad / other |
| `head_clustering()` | K-means clustering of heads by behavioral feature vectors |